# RAG System for SEC 10-K filings
# Summary of Evaluation Retrieval

Measures retrieval quality over **3,996 chunks from ten 10-K filings** against a
60-question gold set (20 easy single-chunk, 20 hard multi-chunk, 20 deliberately unanswerable).

Four retrieval enhancements are built up one stage at a time, each adding a single enhancement to the one before it, so we can attribute every number in the final table to one change:

| stage | enhancement added |
|---|---|
| `dense` | + embeddings only |
| `hybrid` | + BM25, fused with weighted RRF |
| `rerank` | + cross-encoder over the top 50 |
| `decomposed` | + LLM query splitting, round-robin merged |

Then we check does the system correctly **refuse** when the answer isn't in the corpus? This is done using live llm calls for all 60 questions.

**Two metrics.** 
`recall@k` — did at least one correct chunk reach the top k.
`hard_all_gold@k` — did *every* chunk a question needs reach the top k. The second is what hard multi-chunk questions ("which company had higher revenue?") actually require.

Noise Floor: 40 answerable and 20 hard questions so one question is worth 2.5pp on recall and 5pp on all-gold respectively.

## 1. Configuration

In [ ]:

VERSION      = "v2"
GOLD_VERSION = "v3"

CHUNKS_PATH = f"../data/processed/all_chunks_{VERSION}.pkl"
EMB_PATH    = f"../data/processed/embeddings_{VERSION}.npy"
GOLD_PATH   = f"../data/gold/gold_set_60_{GOLD_VERSION}.jsonl"
RESULTS_DIR = "../results"

# bge-reranker-v2-m3 is XLM-R large (8194 max positions). The earlier base model had a
# hard 512-token ceiling covering query AND passage, and XLM-R tokenises this corpus at
# ~1.15x the BGE count - up to ~1.70x on number-dense tables - so 230/3996 chunks were
# being silently truncated. Hence a change of model was made. 
# Tradeoff being longer reranking time
RERANKER          = "BAAI/bge-reranker-v2-m3"
RERANK_MAX_LENGTH = 1024

# reranker model name is in the cache filename
CACHE_PATH = f"../data/processed/rerank_cache_{VERSION}_{RERANKER.split('/')[-1]}.pkl"

# RRF weighted. dense is weighted 3:1 over BM25; RRF_K=60 is the standard smoothing constant.
# Disclaimer: weighted RRF at 3:1 ratio is fitted for the gold set
DENSE_WEIGHT, BM25_WEIGHT = 3.0, 1.0
RRF_K       = 60
FIRST_STAGE = 50      # candidates handed to the cross-encoder

## 2. Corpus

`all_chunks_v2.jsonl` is produced by `src/filings_rag/sec_chunking.py`. Structure-aware: chunks respect Part → Item → section
boundaries and never straddle one.

Each chunk carries a stable `chunk_id` eg. `dell_FY2026_10K:646:1`
(*filing : element index : piece*), so identity survives re-chunking.

### How chunking is specialised for 10-K filings

Generic fixed-window chunking in naive rag treats a filing as flat prose. The problem is that a 10-K is not prose it is a rigidly numbered legal document, roughly half of it tables, wrapped in iXBRL markup.
Cutting it every N words results in information loss in three ways that the chunker addresses.

**1. Document structure is tracked, not inferred.**
The parser walks the HTML maintaining a running `Part → Item → subsection` path, and chunks never cross one of those boundaries. This matters because 10-K section numbering is standardised: Item 1A is always Risk Factors, Item 7 always MD&A, Item 8 always the financial statements. Preserving that path means a chunk knows it is *inside Item 8*, not merely near some accounting words. This also removes the need for contextualisation.

Findings: 
Running page headers are rejected. There a common `Item 7` with no titles, as such we only take specified and titled headers like 
`Item 7. MANAGEMENT'S DISCUSSION AND ANALYSIS` to signal the start to a new section. 
Filings also wrap headings inside single-cell `<table>` elements. So a table containing nothing but a heading is read as a heading. Without that check one filing produced zero section labels.

**2. Every chunk is self-describing.**
A deterministic prefix is prepended to all 3,996 chunks:

```
DELL 10-K FY2026 | FY2026 | Item 8 — FINANCIAL STATEMENTS > Critical Audit Matters
```

The corpus holds three NVIDIA filings whose contents are near-identical year to year. Without company and fiscal period in the embedded text, retrieval cannot tell FY2024 revenue from FY2026 revenue — the surrounding language is the same. 

**3. Tables are serialised so values stay attached to their headers.**
This is the largest difference from generic chunking. A table encodes meaning in two dimensions; a transformer reads one. Flattened to a pipe grid, numbers are seperated from the column header that gives it meaning.

This is a finding in Dantart, A., & Kóvacs-Navarro, M. (2026). Topo-RAG: Topology-aware retrieval for hybrid text–table documents. arXiv:2601.10215. https://arxiv.org/abs/2601.10215

I implemented their solution a header-prepended row serialization, pairing each value inline with its column label. 

```
Columns: Fiscal 2026 | Fiscal 2025 | Fiscal 2024
Revenue: Fiscal 2026 $215,938 | Fiscal 2025 $130,497 | Fiscal 2024 $60,922
```

Three guards fall back to the pipe grid rather than risk corrupting a table and losing information:
when no usable column labels exist, when a label exceeds 200 characters (a sign the header detector has swallowed data rows), and when re-serialisation would drop any numeric token that was present in the source. 


**4. Captions are folded into their tables.**
A heading like *"Revenue by Reportable Segment"* is a separate HTML element from the table beneath it. Left alone it becomes a standalone chunk containing a title and no numbers, which then becomes a distractor, competing with and sometimes outranking the table holding the actual answer.As such, captions are folded into the table that follows them.

**5. iXBRL scaffolding is stripped, its facts kept.**
10K filings (and most financial fillings) are inline-XBRL which has hidden blocks that carry metadata that would otherwise be embedded as text; those are removed, while tagged facts are retained and attached to their chunk.
Eg.`ix:hidden`, `ix:header`, `ix:references`, `ix:resources`

**6. Sized to the embedding model, not to a word count.**
The budget is 430 tokens measured with the embedding model's own tokenizer, so no chunk is silently truncated at encode time. Oversized narrative splits at sentence boundaries; oversized tables split by row group with the header block repeated on every piece, so a continuation fragment is still readable on its own.

### What that produces

| | |
|---|---|
| chunks | 3,996 across 10 filings, 925 distinct sections |
| tokens per chunk | median 255, p95 421, max 430 — **none over budget** |
| carrying the context prefix | 3,996 / 3,996 |
| header-prepended tables | 848 chunks |
| pipe-grid fallback | 304 chunks |
| narrative | 2,844 chunks |
| continuation pieces (`piece_no > 1`) | 424 |

In [38]:
#loading data
import json
import pickle

src = "../data/gold/all_chunks_v2.jsonl"
dst = "../data/processed/all_chunks_v2.pkl"

with open(src, "r", encoding="utf-8") as f:
    v2_chunks = [json.loads(line) for line in f if line.strip()]

assert len(v2_chunks) == 3996

with open(dst, "wb") as f:
    pickle.dump(v2_chunks, f)

print("saved:", len(v2_chunks))

saved: 3996


## 3. Embedding Corpus

Model: `bge-small-en-v1.5`, L2-normalised so a dot product is cosine similarity.

In [24]:
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

with open("../data/processed/all_chunks_v2.pkl", "rb") as f:
    v2_chunks = pickle.load(f)

assert len(v2_chunks) == 3996

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

texts = [c["text"] for c in v2_chunks]

v2_embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

np.save("../data/processed/embeddings_v2.npy", v2_embeddings)

print(v2_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

(3996, 384)


## 4. Load corpus, build BM25, load gold set

BM25 is rebuilt in memory each run as it is fast. Assert ensures corpus and embedding matrix correspond to prevent everything downstream from being based of wrong data

In [25]:
import json, pickle, os, glob
import numpy as np
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

os.makedirs(RESULTS_DIR, exist_ok=True)

with open(CHUNKS_PATH, "rb") as f:
    all_chunks = pickle.load(f)
embeddings = np.load(EMB_PATH)

assert len(all_chunks) == embeddings.shape[0], \
    f"corpus/embeddings mismatch: {len(all_chunks)} vs {embeddings.shape[0]} — re-embed"

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
bm25 = BM25Okapi([c["text"].lower().split() for c in all_chunks])

gold = [json.loads(l) for l in open(GOLD_PATH, encoding="utf-8") if l.strip()]

print(f"{VERSION}: {len(all_chunks)} chunks, {embeddings.shape}, {len(gold)} questions")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

v2: 3996 chunks, (3996, 384), 60 questions


## 5. First stage — dense, BM25, and weighted RRF

Two retrievers with different failure modes. Dense catches paraphrase; BM25 catches exact tokens (ticker symbols, "PricewaterhouseCoopers", literal figures).

**Reciprocal Rank Fusion** merges them by *rank*, not score as the two scoring scales are not comparable, and RRF sidesteps that by having each list contributes `weight / (K + rank)`.

Query vectors and fused pools are cached to prevent recomputing.

In [26]:
#retrieval process with dense and bm25 search. query vector and seach pool cached
_qvec, _pool = {}, {}
RRF_K = 60

def encode_query(query):
    if query not in _qvec:
        _qvec[query] = model.encode([query], normalize_embeddings=True)[0]
    return _qvec[query]

def dense_search(query, k=FIRST_STAGE):
    return list(np.argsort(embeddings @ encode_query(query))[::-1][:k])

def bm25_search(query, k=FIRST_STAGE):
    return list(np.argsort(bm25.get_scores(query.lower().split()))[::-1][:k])

RRF_K = 60
def rrf_weighted(rankings_weights, k=RRF_K):
    scores = defaultdict(float)
    for ranking, weight in rankings_weights:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] += weight / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def fused_pool(query, n=FIRST_STAGE):
    if query not in _pool:
        _pool[query] = rrf_weighted([
            (dense_search(query, n), DENSE_WEIGHT),
            (bm25_search(query, n), BM25_WEIGHT),
        ])
    return _pool[query]

def hybrid_search(query, k=10):
    return fused_pool(query)[:k]

## 6. Cross-encoder reranking

The first stage scores query and chunk *independently* (biencoder) so the chunk's vector is computed before the query exists. A cross-encoder reads the pair together and is far more accurate (supposedly), but far too slow to run over 3,996 chunks, so it only re-scores the top 50 candidates.

Scores are cached to disk because this is the expensive stage: ~70s per query on my CPU.

In [27]:
#reranker, lazy loaded and disk cached to save compute.
#bge-reranker-base is XLM-R base with a hard 512 ceiling covering query AND passage.
#max_length=1024 clears that ceiling before contextualization adds 60-115 more
#XLM-R tokens per chunk. note: ~568M params vs ~278M, so expect a slower warm-up.
_reranker = None
_rerank_cache = {}

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "rb") as f:
        _rerank_cache = pickle.load(f)
    print(f"loaded {len(_rerank_cache)} cached rerank scores")

def get_reranker():
    global _reranker
    if _reranker is None:
        print("loading cross-encoder...")
        _reranker = CrossEncoder(RERANKER, max_length=RERANK_MAX_LENGTH)
    return _reranker

def _rerank_candidates(query):
    if query not in _rerank_cache:
        candidates = fused_pool(query)[:FIRST_STAGE]
        pairs = [(query, all_chunks[i]["text"]) for i in candidates]
        raw = np.asarray(
            get_reranker().predict(pairs, batch_size=len(pairs)),
            dtype=float,
        )
        _rerank_cache[query] = (list(candidates), raw)
    return _rerank_cache[query]

def search_reranked(query, k=10):
    candidates, raw = _rerank_candidates(query)
    return [candidates[i] for i in np.argsort(raw)[::-1][:k]]

loaded 102 cached rerank scores


## 7. Query decomposition

The failure this exists to fix hard questions where multiple chunks need to be retrieved and the info compared. Upon inspecting the failures for hard questions like *"which company had higher revenue, Meta or NVIDIA?"*, the reranked top-10 contained **zero Meta chunks** and **only** Nvidia chunks. One entity monopolised every slot, so `all_gold` was unreachable no matter how good the ranking was.

A cheap model (Haiku) is used to split similar query into self-contained sub-queries, each searched separately. The LLM call is cached to disk keyed on question text alone, so the cache is corpus-independent and survives re-chunking.

Problem: Anthropic 1.0.0 `Messages.create` no longer takes temperature = 1.0 as a parameter, so determinism is based on my cached query which I checked through manually. It is not feasible to check every query at scale and I did not test whether decomposition will fail queries outside of the gold set. As such, there is risk of overfitting and the assumption that decomposition works 100% of the time.

In [28]:
#Query decomposition.
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv("../.env")
_client = Anthropic()

DECOMP_CACHE = "../data/processed/decomp_cache.pkl"
DECOMP_MODEL = "claude-haiku-4-5-20251001"

_decomp_cache = {}
if os.path.exists(DECOMP_CACHE):
    with open(DECOMP_CACHE, "rb") as f:
        _decomp_cache = pickle.load(f)
    print(f"loaded {len(_decomp_cache)} cached decompositions")

DECOMP_PROMPT = """You are preparing search queries for a retrieval system over SEC 10-K filings.

Question: {question}

If answering this requires facts from two or more separate places - different companies,
different fiscal years, or different sections - split it into independent sub-queries.
Each sub-query must be self-contained and name its company and fiscal period explicitly.

If the question asks for a single fact from one place, return it unchanged as a single item.

Output only a JSON array of strings. No preamble, no markdown fences."""


def _strip_fences(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
        if raw.rstrip().endswith("```"):
            raw = raw.rstrip()[:-3]
    return raw.strip()


def decompose(question):
    if question in _decomp_cache:
        return _decomp_cache[question]

    resp = _client.messages.create(
        model=DECOMP_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": DECOMP_PROMPT.format(question=question)}],
    )

    try:
        subs = json.loads(_strip_fences(resp.content[0].text))
        ok = (isinstance(subs, list) and subs
              and all(isinstance(s, str) and s.strip() for s in subs))
        if not ok:
            subs = [question]
    except json.JSONDecodeError:
        subs = [question]

    _decomp_cache[question] = subs
    return subs

loaded 60 cached decompositions


## 8. Decomposed search — round-robin merge

Round-robin where two sub-queries at k=10, each contributes roughly its top 5 (k/n) so no single company can occupy every slot. 

An atomic question is searched with the *original* wording, not the model's paraphrase, so `rerank` and `decomposed` differ only on questions that
genuinely split.

In [29]:
def search_decomposed(query, k=10):
    subs = decompose(query)

    # DECOMP_PROMPT says to return an atomic question unchanged, but the model
    # still paraphrases ("What was" -> "What were"). Reranking a paraphrase would
    # make this config differ from the rerank baseline on questions that never
    # decomposed. We use only the original so the only difference between
    # `rerank` and `decomposed` is the questions that actually split.
    if len(subs) == 1:
        return search_reranked(query, k)

    per_sub = [search_reranked(s, k) for s in subs]

    merged, seen = [], set()
    for rank in range(k):
        for lst in per_sub:
            if rank < len(lst) and lst[rank] not in seen:
                seen.add(lst[rank])
                merged.append(lst[rank])
                if len(merged) == k:
                    return merged

    # heavy overlap between sub-queries can leave the merge short of k. top up more chunks from
    # the undecomposed query so every config is scored on the same number of results.
    if len(merged) < k:
        for i in search_reranked(query, k):
            if i not in seen:
                seen.add(i)
                merged.append(i)
                if len(merged) == k:
                    break

    return merged

### Sanity check on the split

Two API calls. 
Expect two sub-queries for the comparison and one unchanged for the atomic question.

In [30]:
for _q in [
    "Between Meta's fiscal year 2025 and NVIDIA's fiscal year 2026, which company reported higher total revenue?",
    "What was Apple's total net sales for fiscal year 2025?",
]:
    print(_q)
    for _s in decompose(_q):
        print("   ->", _s)
    print()

Between Meta's fiscal year 2025 and NVIDIA's fiscal year 2026, which company reported higher total revenue?
   -> Meta total revenue fiscal year 2025
   -> NVIDIA total revenue fiscal year 2026

What was Apple's total net sales for fiscal year 2025?
   -> What was Apple's total net sales for fiscal year 2025?



## 9. Decompose first, then price the rerank job

Order matters since decomposition is cheap and cached but reranking is ~70 s/query. Running decomposition first means a broken prompt or an over-split question surfaces *here* and can be checked

`evaluate()` only ever searches answerable questions.

In [ ]:
from collections import Counter

for i, q in enumerate(gold):
    decompose(q["question"])
    print(f"decompose {i+1}/{len(gold)}", end="\r")

with open(DECOMP_CACHE, "wb") as f:
    pickle.dump(_decomp_cache, f)

answerable = [q for q in gold if q["gold_chunks"]]
print(f"\n{len(gold)} questions, {len(answerable)} answerable")
print("sub-queries per question:",
      dict(sorted(Counter(len(decompose(q["question"])) for q in gold).items())))

to_rerank = {q["question"] for q in answerable}
for q in answerable:
    subs = decompose(q["question"])
    if len(subs) > 1:
        to_rerank.update(subs)

to_rerank = sorted(to_rerank)
missing = [s for s in to_rerank if s not in _rerank_cache]

print(f"\nevaluate() needs {len(to_rerank)} queries reranked "
      f"({len(to_rerank) - len(missing)} already cached, {len(missing)} to compute)")


decompose 60/60
60 questions, 40 answerable
sub-queries per question: {1: 28, 2: 26, 3: 2, 4: 3, 10: 1}

evaluate() needs 81 queries reranked (81 already cached, 0 to compute)
estimated rerank warm-up: 0 min


## 10. Warming the rerank cache

In [32]:
#one rerank pass over exactly what evaluate() will ask for.
import time

t0 = time.time()
for i, s in enumerate(missing):
    _rerank_candidates(s)
    print(f"rerank {i+1}/{len(missing)}   {(time.time()-t0)/60:5.1f} min elapsed", end="\r")

with open(CACHE_PATH, "wb") as f:
    pickle.dump(_rerank_cache, f)

print(f"\ncached {len(_rerank_cache)} queries -> {CACHE_PATH}")
print(f"took {(time.time()-t0)/60:.1f} min")


cached 102 queries -> ../data/processed/rerank_cache_v2_bge-reranker-v2-m3.pkl
took 0.0 min


## 11. Metrics

`recall@k` / `r@k` — at least one gold chunk in the top k.

`hard_all_gold@k` / `allgold@k` — *every* gold chunk in the top k, over the hard subset only.

`easy@k and hard@k` for the easy and hard questions respectively

In [33]:
#metrics to evaluate recall k
def evaluate(search_fn, ks=(1, 5, 10, 20)):
    answerable = [q for q in gold if q["gold_chunks"]]
    hard       = [q for q in gold if q["type"] == "hard" and q["gold_chunks"]]
    easy       = [q for q in gold if q["type"] == "easy" and q["gold_chunks"]]

    full = {q["id"]: search_fn(q["question"], max(ks)) for q in answerable}

    m = {"n_answerable": len(answerable), "n_hard": len(hard), "n_easy": len(easy)}

    for k in ks:
        ret = {qid: set(r[:k]) for qid, r in full.items()}

        for label, subset in (("", answerable), ("easy_", easy), ("hard_", hard)):
            hits = sum(1 for q in subset if ret[q["id"]] & set(q["gold_chunks"]))
            m[f"{label}recall@{k}"] = round(hits / len(subset), 3)
            m[f"{label}recall@{k}_hits"] = hits

        complete = sum(1 for q in hard if set(q["gold_chunks"]).issubset(ret[q["id"]]))
        m[f"hard_all_gold@{k}"] = round(complete / len(hard), 3)
        m[f"hard_all_gold@{k}_hits"] = complete

    return m


def run_and_save(search_fn, config_name, config_dict):
    metrics = evaluate(search_fn)
    payload = {
        "version": VERSION,
        "gold_version": GOLD_VERSION,
        "config_name": config_name,
        "n_chunks": len(all_chunks),
        "config": config_dict,
        "metrics": metrics,
    }
    path = f"{RESULTS_DIR}/{VERSION}_{config_name}.json"
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"saved {path}")
    return payload

## 12. Run all four configurations

In [34]:
#running dense, bm25 and reranked search individually. remember to warm cache.
#each call is bound to _ so the notebook does not auto-display the returned payload -
#the last bare expression in a cell renders its whole metrics dict, which is ~30 keys
#of noise directly above the results table that formats them properly.
_ = run_and_save(dense_search, "dense",
             {"method": "dense", "model": "BAAI/bge-small-en-v1.5"})

_ = run_and_save(hybrid_search, "hybrid",
             {"method": "weighted_rrf", "dense_weight": DENSE_WEIGHT,
              "bm25_weight": BM25_WEIGHT, "rrf_k": RRF_K})

_ = run_and_save(search_reranked, "rerank",
             {"method": "weighted_rrf + cross-encoder",
              "reranker": RERANKER, "rerank_max_length": RERANK_MAX_LENGTH,
              "first_stage": FIRST_STAGE})

_ = run_and_save(search_decomposed, "decomposed",
             {"method": "decompose + weighted_rrf + cross-encoder",
              "decomp_model": DECOMP_MODEL,
              "reranker": RERANKER,
              "rerank_max_length": RERANK_MAX_LENGTH,
              "first_stage": FIRST_STAGE})

saved ../results/v2_dense.json
saved ../results/v2_hybrid.json
saved ../results/v2_rerank.json
saved ../results/v2_decomposed.json


## 13. Results

In [35]:
#comparison table
rows = []
for path in sorted(glob.glob(f"{RESULTS_DIR}/*.json")):
    r = json.load(open(path))
    if "config_name" not in r:
        continue
    m = r["metrics"]
    rows.append({
        "run": f"{r['version']}_{r['config_name']}",
        "chunks": r["n_chunks"],
        "r@1": m["recall@1"], "r@5": m["recall@5"],
        "r@10": m["recall@10"], "r@20": m["recall@20"],
        "easy@1": m["easy_recall@1"],
        "hard@10": m["hard_recall@10"],
        "allgold@10": m["hard_all_gold@10"],
        "allgold@20": m["hard_all_gold@20"],
    })

pd.DataFrame(rows).set_index("run")

,chunks,r@1,r@5,r@10,r@20,easy@1,hard@10,allgold@10,allgold@20
run,,,,,,,,,
naive_dense,3173,0.200,0.425,0.475,0.600,0.20,0.50,0.05,0.10
v2_decomposed,3996,0.375,0.675,0.825,0.950,0.40,0.75,0.45,0.55
v2_dense,3996,0.225,0.400,0.650,0.850,0.25,0.60,0.25,0.40
v2_hybrid,3996,0.225,0.425,0.650,0.875,0.25,0.60,0.15,0.40
v2_rerank,3996,0.300,0.575,0.775,0.850,0.40,0.65,0.15,0.25


### What the results show when tested against the specific gold set
**How we treat the corpus data mattered the most**

The single largest jump comes from the addition of structure aware chunking.Naive had fewer chunks (3,173 vs 3,996), meaning fewer distractors and in theory should perform better, but still lost on every metric against v2_dense.

**Adding BM25 gave mixed results**

From v2_dense to v2_hybrid (the weighted RRF of dense + bm25), we achieved slight recall gain, but allgold@10 got worse.
When RRF was weighted 1:1, BM25 unexpectedly degraded retrieval quality.
The hypothesis here is due to corpus homogeneity since 10 filings written to the same regulatory template, supposedly rare terms like "total revenue" and "fiscal year" appear in nearly every document, causing BM25’s inverse document frequency (IDF) to fail. 

Research on this is mixed:

Saber Zerhoudi, Adam Roegiest, Jelena Mitrovic, Michael Granitzer As We May Search. (2026). arXiv:2606.29652. https://arxiv.org/abs/2606.29652 claims that on FiQA (financial dataset), the BM25 component dilutes strong dense signals, and pure dense wins. 

Akarsu, M., Karaman, R. K., & Mierbach, C. (2026). From BM25 to Corrective RAG: Benchmarking Retrieval Strategies for Text-and-Table Documents. arXiv:2604.01733. https://arxiv.org/abs/2604.01733 claims BM25 helps slightly.

**Adding reranking also gave mixed results**

Reranking is very good at pushing the single best chunk to the top. Biggest gains on easy@1 and r@1, but allgold@20 dropped. Reranking helped single fact questions but was weak in hard questions.
The hypothesis here is table-vs-prose bias for cross-encoding as when inspecting the actual failures, many were adjacent chunks and the chunks ranked higher were from the correct section but the actual data in tables was not present.

**Decomposition is effective for hard questions**

Splitting multi-chunk questions into sub-queries helped leverage reranking's effectiveness against easy single fact queries. This kept easy@1 unchanged and tripled allgold@10 (0.15 to 0.45) from v2_rerank to v2_decomposed.
Comparing v2_decomposed with the naive_dense RAG on an identical corpus, embedding model, and gold set, r@20 increased from 0.60 to 0.95 and allgold@10 from 0.05 to 0.45.

## 14. Refusal

The 20 unanswerable questions have no correct answer in the corpus so the only correct answer is to refuse answering.

Two error types: a **false refusal** loses a real answer, a **missed refusal** produces a confident wrong one. Both are reported.

Generation is the only stochastic step in the pipeline, so it runs three times over all 60 questions — **180 calls** — and the table reports the spread. 

In [36]:
#UNANSWERABLE - generation refusal. recall@1 of 0.375 means the generator must work from imperfect
#ranking, so the prompt says so explicitly rather than pretending the top-k is gold.
GEN_MODEL = "claude-sonnet-5"   

GEN_PROMPT = """You are answering questions about SEC 10-K filings using retrieved excerpts.

The excerpts were retrieved automatically. Some may be irrelevant, incomplete, or from
the wrong company or fiscal year. They may not contain the answer at all.

Excerpts:
{context}

Question: {question}

Rules:
- Answer only from the excerpts. Do not use outside knowledge.
- If the excerpts do not contain enough information, set answerable to false and say
  what is missing. Do not guess, approximate, or infer from a related figure.
- A figure for a different company, segment, or fiscal year is not an answer.

Return JSON only: {{"answerable": true or false, "answer": "<answer, or what is missing>"}}"""


def generate(question, search_fn=None, k=20):
    search_fn = search_fn or search_reranked
    ctx = "\n\n---\n\n".join(all_chunks[i]["text"] for i in search_fn(question, k))
    resp = _client.messages.create(
        model=GEN_MODEL,
        max_tokens=1000,
        messages=[{"role": "user",
                   "content": GEN_PROMPT.format(context=ctx, question=question)}],
    )
    text = next((b.text for b in resp.content if b.type == "text"), "")
    try:
        out = json.loads(_strip_fences(text))
        return bool(out.get("answerable")), str(out.get("answer", ""))[:400]
    except (json.JSONDecodeError, AttributeError):
        return True, text[:400]      # unparseable = did not refuse

In [ ]:
#180 generation calls: 3 runs x 60 questions.
#generate() defaults to search_reranked at k=20

N_RUNS = 3

ans_q = [q for q in gold if q["gold_chunks"]]        # 40 answerable
una_q = [q for q in gold if not q["gold_chunks"]]    # 20 unanswerable

runs = []
for r in range(N_RUNS):
    rec = {}
    for i, q in enumerate(gold):
        rec[q["id"]] = generate(q["question"])      
        print(f"run {r+1}/{N_RUNS}  question {i+1}/{len(gold)}", end="\r")
    runs.append(rec)
print(" " * 40, end="\r")

rows = []
for i, rec in enumerate(runs, 1):
    refused = sum(1 for q in una_q if not rec[q["id"]][0])       # correct refusals
    false   = sum(1 for q in ans_q if not rec[q["id"]][0])       # wrongly refused
    rows.append({"run": str(i),
                 "refused": refused, "refusal_rate": round(refused / len(una_q), 3),
                 "false_refused": false, "false_refusal_rate": round(false / len(ans_q), 3)})

df_refusal = pd.DataFrame(rows)
mean = df_refusal.mean(numeric_only=True).round(3)
mean["run"] = "mean"
df_refusal = pd.concat([df_refusal, mean.to_frame().T], ignore_index=True).set_index("run")

print(f"refused = correct refusals out of {len(una_q)} unanswerable")
print(f"false_refused = answerable questions wrongly refused, out of {len(ans_q)}")

#a hallucination is an unanswerable question answered anyway - the failure that matters
#most. silent when there are none.
for q in una_q:
    hit = [r for r in runs if r[q["id"]][0]]
    if hit:
        print(f"\nNOT REFUSED [{q['id']}] {len(hit)}/{N_RUNS} runs: {q['question'][:60]}")
        print(f"   -> {hit[0][q['id']][1][:140]}")

os.makedirs(f"{RESULTS_DIR}/refusal", exist_ok=True)
with open(f"{RESULTS_DIR}/refusal/{VERSION}_refusal.json", "w") as f:
    json.dump({
        "version": VERSION, "gold_version": GOLD_VERSION, "config_name": "refusal",
        "n_chunks": len(all_chunks),
        "config": {"gen_model": GEN_MODEL, "retrieval": "search_reranked", "k": 20,
                   "n_runs": N_RUNS, "reranker": RERANKER},
        "metrics": df_refusal.to_dict(),
    }, f, indent=2)

df_refusal

refused = correct refusals out of 20 unanswerable
false_refused = answerable questions wrongly refused, out of 40

NOT REFUSED [q060] 3/3 runs: Which of these ten companies has committed to the earliest c
   -> Microsoft, which committed to becoming carbon negative (its carbon-neutrality-related goal) by 2030.


,refused,refusal_rate,false_refused,false_refusal_rate
run,,,,
1,19,0.95,7,0.175
2,19,0.95,7,0.175
3,19,0.95,7,0.175
mean,19.0,0.95,7.0,0.175


### What the refusal statistics show:
High false refusal rate are due to hard multi-chunk questions. This is likely linked to allgold@20=0.55 and not 100%. This is a retriveal issue, not a generation failure since incomplete information was provided, the model rightfully refused. 

Q60 is a deliberately hard unanswerable question as Microsoft's real carbon-negative-by-2030 commitment exists in the corpus, but nothing establishes it's the earliest among all ten companies. 

One interesting thing that deferred from previous runs. Here, the model sets k=20, meaning the top 20 chunks are fed to the llm. 
In previous k=10 runs, this was correctly refused, which is a variance not reflected here. The hypothesis is that as k doubled, the amount of distractors increased, likely microsoft's carbon neutral goal entered the mix, and the model did not refuse correctly. 

---

## Summary

| | |
|---|---|
| Corpus | 3,996 chunks, 10 filings, structure-aware, header-prepended tables |
| Embeddings | `BAAI/bge-small-en-v1.5` |
| Fusion | weighted RRF, dense 3 : BM25 1, k=60 |
| Reranker | `BAAI/bge-reranker-v2-m3`, max_length 1024, top 50 |
| Decomposition | `claude-haiku-4-5`, round-robin merge |
| Generation | `claude-sonnet-5` |

**Best configuration:** `decomposed` — recall@20 0.950 (38/40), all_gold@20 0.550
(11/20), with 19/20 correct refusals and zero hallucinated figures.
